# Phase 1: Exploratory Data Analysis

This notebook inspects the support ticket dataset. It does not train a machine learning model.

We will use `Issue_Category` as the target (the value we may want to predict).This notebook reads `customer_support_tickets.csv`.

## 1. Load the data

Why: We need to load the table before we can inspect its size, columns, and contents. Keeping the file path in one variable makes it easy to change.

In [ ]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/raw/customer_support_tickets.csv")
tickets = pd.read_csv(data_path)

tickets.head()

## 2. Number of rows and columns

Why: The dataset size gives us a basic sense of how much data is available and confirms that the file loaded as expected.

In [ ]:
rows, columns = tickets.shape
print(f"Rows: {rows}")
print(f"Columns: {columns}")

## 3. Column names and data types

Why: Column names tell us what information is available. Data types show how pandas currently interprets each column; dates may load as text and can be converted later if needed.

In [ ]:
print("Column names:")
print(tickets.columns.tolist())

print("\nData types:")
print(tickets.dtypes)

## 4. Missing values

Why: Missing values can affect later analysis and model inputs. This check counts empty cells in each column.

In [ ]:
# Findd none values and get the some
missing_values = tickets.isna().sum()
missing_values[missing_values > 0]

## 5. Target category values and distribution

Why: We need to know which categories exist and whether some are much more common than others. A highly uneven distribution can affect how we interpret a future classifier.

In [ ]:
target_column = "Issue_Category" # it is the resolution

category_counts = tickets[target_column].value_counts(dropna=False)
category_percentages = tickets[target_column].value_counts(normalize=True, dropna=False).mul(100)

category_summary = pd.DataFrame({
    "count": category_counts,
    "percent": category_percentages.round(1),
})
category_summary

In [ ]:
import matplotlib.pyplot as plt

# display the numbers
category_counts.plot(kind="bar", color="steelblue")
plt.title("Tickets by issue category")
plt.xlabel("Issue category")
plt.ylabel("Number of tickets")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Which columns may help classify a ticket?

Why: Classification should use information available when the ticket arrives. Ticket text is directly about the issue; operational and personal details may not describe the issue or may not be available at prediction time.

In [ ]:
tickets.columns.tolist()

Likely useful at ticket arrival:

- `Ticket_Description`: describes the customer's problem and is a strong candidate input.
- `Ticket_Subject`: may contain useful issue wording, but inspect it carefully for shortcuts or category-specific templates.
- `Ticket_Channel`: available at arrival and may provide a small amount of context.
- `Submission_Date`: available at arrival, though date patterns may not generalize and should be justified before use.

`Priority_Level` may be available at arrival, but it is a separate assessment that could overlap with category. Whether to use it depends on how it is assigned in a real support workflow. `Ticket_ID`, customer name, and email are generally not useful descriptions of the issue.

## 7. Possible data leakage

Why: Data leakage happens when the inputs contain information that would not be available at the time we want to make a prediction, or contain the answer through a shortcut. A model can then look strong without learning a useful pattern.

- `Resolution_Time_Hours` and `Satisfaction_Score` are likely recorded after the ticket is handled, so they should not be used to classify a new incoming ticket.
- `Assigned_Agent` may be chosen after triage and could reflect the issue category. Confirm when assignment happens before using it.
- `Ticket_Subject` deserves special review: subjects appear strongly associated with `Issue_Category` in this dataset. It might be genuinely useful text, but could also encode category templates or a shortcut. Compare a description-only baseline with subject text in a later step.
- `Priority_Level` might be set at intake or later. Check its timing and process before deciding whether it is safe.

We need to exmine these columns.